# 08 · Chunking 高级

> 基础切分只保证“长度合适”，不保证“一块讲一件事”。高级切分让 chunk 语义内聚，显著提升检索命中率。

**本文件覆盖知识点**：Semantic Chunking / Sentence Window / Parent-Child / Hierarchical / Recursive / Contextual Chunking / Proposition / Document-Structure / Markdown / Code / Table Chunking

In [ ]:
# ===== 本课共用：真调 LLM 做「说明 / 演示」的小助手 =====
# 凡某个知识点能靠“真调一次大模型”当场讲清 / 演示的，下面的 cell 都用 _llm_live()
# 真调 qwen-plus 并打印模型输出作为说明；只有在项目根 .env 配了 DASHSCOPE_API_KEY 时才真调，
# 没配置就打印一段固定的演示样例，保证整个 notebook 不联网也能完整读下来。
from dotenv import load_dotenv; load_dotenv()
import os
from dashscope import Generation

_KEY = os.getenv('DASHSCOPE_API_KEY', '').strip()
_HAS_KEY = bool(_KEY) and '你的' not in _KEY

def _llm_live(prompt, fallback, system='你是资深 RAG 讲师，回答精炼、结构清晰、尽量结合例子。', temperature=0.3, model='qwen-plus'):
    """真调一次 qwen-plus 并打印结果；无 Key 时打印 fallback 作为演示样例。返回模型文本或 None。"""
    if not _HAS_KEY:
        print('未在 .env 配置 DASHSCOPE_API_KEY，跳过实时调用。以下是固定演示样例（配置后自动变为实时输出）：')
        print(fallback)
        return None
    msgs = [{'role': 'system', 'content': system}, {'role': 'user', 'content': prompt}]
    try:
        r = Generation.call(model=model, messages=msgs, temperature=temperature, result_format='message', api_key=_KEY)
        if r.status_code == 200:
            text = r.output.choices[0].message.content
            print('—— 模型实时输出 ——')
            print(text)
            return text
        print('调用失败：', getattr(r, 'code', ''), getattr(r, 'message', ''))
    except Exception as e:
        print('调用异常：', e)
    print('fallback：')
    print(fallback)
    return None


In [ ]:
# ===== 本课共用：真实检索底座 =====
# 真语料(data/) → 真切分 → 真向量(text-embedding-v3) → 真索引(FAISS + BM25)
# → 真重排(qwen3-rerank) → 真生成(qwen-plus)。各课在这个底座上演示自己的知识点。
#
# 说明：向量按内容哈希缓存在 .cache/emb.npz（首次真调、之后复用，避免反复花 token）。
# 没配 DASHSCOPE_API_KEY 时仍可用：向量直接从缓存读（是此前真实调用的结果），
# 但需要现场调用模型的重排/生成会打印录制结果并提示配置方式。
from dotenv import load_dotenv; load_dotenv()
import os, re, json, time, hashlib
from pathlib import Path
import numpy as np

_KEY = os.getenv('DASHSCOPE_API_KEY', '').strip()
_HAS_KEY = bool(_KEY) and '你的' not in _KEY
_DATA = Path('data') if Path('data').is_dir() else Path.cwd() / 'data'
_CACHE_FILE = Path('.cache') / 'emb.npz'
EMBED_MODEL = 'text-embedding-v3'
RERANK_MODEL = 'qwen3-rerank'
NO_KEY_TIP = ('未配置 DASHSCOPE_API_KEY：需要现场调用模型的部分将展示此前真实调用的录制结果，'
              '在项目根 .env 配置后自动变为实时调用。')

def recorded(text, note=''):
    """无 Key 时展示「此前真实运行的录制结果」。内容来自真实调用，不是编造的假数据。"""
    print(NO_KEY_TIP)
    print('—— 录制结果%s ——' % ('（' + note + '）' if note else ''))
    print(text)

if not _HAS_KEY:
    print(NO_KEY_TIP)

# ---------- 1) 语料：读 data/ 全部 Markdown，按小节切块 ----------
# 注意：评测集.md 是「人工标注的答案」，不能进索引 —— 否则第 34 课评测时，
# 标注本身会被检索命中，指标虚高（数据泄漏）。这里显式排除。
_EXCLUDE = {'评测集.md'}

def load_chunks(chunk_size=300, overlap=60):
    """按「## 小节」切分，小节过长再按句子窗口滑切。返回 [{'i','text','source','section'}]"""
    out = []
    for p in sorted(_DATA.glob('*.md')):
        if p.name in _EXCLUDE:
            continue
        section, buf = p.stem, []
        for line in p.read_text(encoding='utf-8').splitlines():
            if line.startswith('## '):
                if buf: out += _split_section(buf, section, p.name, chunk_size, overlap)
                section, buf = line[3:].strip(), [line]
            elif line.startswith('# '):
                section = line[2:].strip()
            else:
                buf.append(line)
        if buf: out += _split_section(buf, section, p.name, chunk_size, overlap)
    for i, c in enumerate(out):
        c['i'] = i
    return out

def _split_section(lines, section, source, chunk_size, overlap):
    """小节内容按句号聚合成 ~chunk_size 字的片段，相邻片段留 overlap 字重叠"""
    text = '\n'.join(lines).strip()
    if not text: return []
    sents = [s for s in re.split(r'(?<=[。！？\n])', text) if s.strip()]
    chunks, buf = [], ''
    for s in sents:
        if len(buf) + len(s) > chunk_size and buf:
            chunks.append(buf.strip())
            buf = buf[-overlap:] + s          # 保留尾部 overlap 字做上下文重叠
        else:
            buf += s
    if buf.strip(): chunks.append(buf.strip())
    return [{'text': c, 'source': source, 'section': section} for c in chunks]

# ---------- 2) 向量：真调 text-embedding-v3（分批 + 重试 + 内容哈希缓存）----------
def _load_cache():
    if not _CACHE_FILE.exists():
        return {}
    try:
        z = np.load(_CACHE_FILE, allow_pickle=False)
        return dict(zip(z['hashes'].tolist(), z['vectors']))
    except Exception as e:                      # 文件损坏（例如多进程同时写）：当空缓存重建，别让 notebook 挂掉
        print('向量缓存不可读(%s: %s)，将重新向量化：%s' % (type(e).__name__, e, _CACHE_FILE))
        return {}

def _save_cache(cache):
    """写盘前先与磁盘上已有内容合并，再原子替换 —— 避免多个进程同时跑时互相覆盖 / 写坏文件"""
    _CACHE_FILE.parent.mkdir(parents=True, exist_ok=True)
    for k, v in _load_cache().items():
        cache.setdefault(k, v)
    hs = np.array(list(cache.keys()))
    vs = np.array([cache[h] for h in cache.keys()], dtype='float32')
    # 进程号唯一，别抢同一个临时文件；注意 np.savez_compressed 会自动补 .npz 后缀，临时名必须也是 .npz 结尾
    tmp = _CACHE_FILE.with_name('%s.%d.tmp.npz' % (_CACHE_FILE.stem, os.getpid()))
    np.savez_compressed(tmp, hashes=hs, vectors=vs)
    try:
        os.replace(tmp, _CACHE_FILE)            # 原子替换：别的进程读到的永远是完整文件
    except OSError:                             # 目标被占用时稍等再试
        time.sleep(0.2); os.replace(tmp, _CACHE_FILE)

def _key(text, model):
    return hashlib.sha1((model + '\x00' + text).encode('utf-8')).hexdigest()[:16]

def embed(texts, model=EMBED_MODEL, batch=10):
    """真调 Embedding；命中缓存则直接用（缓存来自真实调用）。返回已 L2 归一化的向量"""
    if isinstance(texts, str): texts = [texts]
    cache, todo = _load_cache(), []
    for t in texts:
        k = _key(t, model)
        if k not in cache and k not in [x[0] for x in todo]:
            todo.append((k, t))
    if todo and not _HAS_KEY:
        raise RuntimeError('本地缓存缺少 %d 条向量，且未配置 DASHSCOPE_API_KEY，无法现场向量化。'
                           '请在项目根 .env 配置 Key 后重跑。' % len(todo))
    if todo:
        from dashscope import TextEmbedding
        pending = todo
        while pending:                                  # 批次过大就减半重试
            b = pending[:batch]
            r = TextEmbedding.call(model=model, input=[t for _, t in b], api_key=_KEY)
            if r.status_code == 200:
                for (k, _), e in zip(b, sorted(r.output['embeddings'], key=lambda e: e['text_index'])):
                    cache[k] = np.array(e['embedding'], dtype='float32')
                pending = pending[len(b):]
            elif batch > 1:
                batch //= 2
            else:
                raise RuntimeError('向量化失败: %s %s' % (r.code, r.message))
        _save_cache(cache)
    v = np.array([cache[_key(t, model)] for t in texts], dtype='float32')
    return v / (np.linalg.norm(v, axis=1, keepdims=True) + 1e-10)

# ---------- 3) 索引：FAISS（归一化后内积=余弦）+ BM25 ----------
import faiss
from rank_bm25 import BM25Okapi

def tokenize(text):
    """中文用「单字 + 相邻双字」切词，无需外部分词器（与第 16 课一致）"""
    t = re.sub(r'\s+', '', text)
    return [t[i] for i in range(len(t))] + [t[i:i + 2] for i in range(len(t) - 1)]

CHUNKS = load_chunks()
VECS = embed([c['text'] for c in CHUNKS])
INDEX = faiss.IndexFlatIP(VECS.shape[1]); INDEX.add(VECS)
BM25 = BM25Okapi([tokenize(c['text']) for c in CHUNKS])
print('语料就绪：%d 篇文档 → %d 个片段，向量维度 %d' % (len({c['source'] for c in CHUNKS}), len(CHUNKS), VECS.shape[1]))

# ---------- 4) 检索：稠密 / 稀疏 / 混合（RRF 融合）----------
def dense_retrieve(query, k=5):
    sims, ids = INDEX.search(embed(query), k)
    return [dict(CHUNKS[i], score=float(s), from_='dense') for i, s in zip(ids[0], sims[0]) if i != -1]

def sparse_retrieve(query, k=5):
    scores = BM25.get_scores(tokenize(query))
    top = np.argsort(-scores)[:k]
    return [dict(CHUNKS[i], score=float(scores[i]), from_='bm25') for i in top if scores[i] > 0]

def hybrid_retrieve(query, k=5, rrf_k=60, pool=10):
    """RRF 融合：score = Σ 1/(rrf_k + rank)，只用名次不用原始分数，天然可比"""
    fused = {}
    for name, hits in (('dense', dense_retrieve(query, pool)), ('bm25', sparse_retrieve(query, pool))):
        for rank, h in enumerate(hits, 1):
            cur = fused.setdefault(h['i'], dict(h, score=0.0, from_=set()))
            cur['score'] += 1.0 / (rrf_k + rank)
            cur['from_'].add(name)
    return sorted(fused.values(), key=lambda x: -x['score'])[:k]

# ---------- 5) 重排：真调 DashScope TextReRank ----------
def rerank(query, docs, top_n=3, model=RERANK_MODEL):
    """docs 可以是字符串列表或检索结果 dict 列表；返回 [(文档, 相关性分数)]"""
    texts = [d['text'] if isinstance(d, dict) else d for d in docs]
    if not texts: return []
    if not _HAS_KEY:
        print(NO_KEY_TIP); return [(t, None) for t in texts[:top_n]]
    from dashscope import TextReRank
    r = TextReRank.call(model=model, query=query, documents=texts,
                        top_n=min(top_n, len(texts)), return_documents=False, api_key=_KEY)
    if r.status_code != 200:
        raise RuntimeError('重排失败: %s %s' % (r.code, r.message))
    return [(texts[it['index']], float(it['relevance_score'])) for it in r.output['results']]

# ---------- 6) 生成：qwen-plus（带重试）+ 结构化 JSON 输出 ----------
def chat(prompt, system='你是严谨的 RAG 助手：只依据给定资料回答，资料里没有的就直说不知道。',
         temperature=0.3, model='qwen-plus', retries=3):
    if not _HAS_KEY:
        return None
    from dashscope import Generation
    for attempt in range(retries):
        r = Generation.call(model=model, messages=[{'role': 'system', 'content': system},
                                                   {'role': 'user', 'content': prompt}],
                            temperature=temperature, result_format='message', api_key=_KEY)
        if r.status_code == 200:
            return r.output.choices[0].message.content
        if attempt == retries - 1:
            raise RuntimeError('生成失败: %s %s' % (r.code, r.message))
        time.sleep(1.5 * (attempt + 1))          # 限流类错误退避重试
    return None

def chat_json(prompt, system='只输出 JSON，不要任何解释或代码块标记。', retries=2, **kw):
    """要求模型输出 JSON 并解析；解析失败时把报错回喂再试一次"""
    for attempt in range(retries + 1):
        out = chat(prompt, system=system, **kw)
        if out is None: return None
        seg = out[out.find('{'): out.rfind('}') + 1]     # 容忍 ```json 包裹与前后废话
        try:
            return json.loads(seg)
        except Exception as e:
            if attempt == retries: raise
            prompt = prompt + '\n\n上次输出无法解析(%s)，请只输出合法 JSON。' % e
    return None


## 1. 为什么需要语义化切分

```text
差:  chunk1=[....Redis采用单线程模]
     chunk2=[型...]          ← 一个概念被切成两半，谁都不完整
好:  chunk=[Redis 采用单线程事件循环模型，负责处理命令请求]  ← 一个语义单元
```

目标：**一个 chunk ≈ 一个相对独立的语义单元**（一条事实、一个论点、一段说明）。

## 2. 语义类切分（一句话理解）

| 方法 | 一句话 | 何时用 |
|------|--------|--------|
| **Semantic Chunking** | 把句子先向量化，按“语义突变点”切 | 主题变化明显的长文 |
| **Proposition Chunking** | 用 LLM 把文本拆成“最小事实陈述”（每条只含一个断言） | 高精度细粒度问答 |
| **Contextual Chunking** | 为每个 chunk 附一段“它处于什么上下文”的说明 | 碎片缺上下文（第 24 课） |
| **Sentence Window** | 按句存，检索时取前后 N 句 | 长文档（第 24 课） |
| **Parent-Child** | 父块大、子块小，检索子块返回父块 | 细召回+大上下文（第 24 课） |

Semantic Chunking 的核心是找“语义断点”：相邻句子的向量相似度突然变低，说明话题切换了，在此处切。

In [ ]:
# 语义切分实证：对真实语料算「相邻句相似度」，再按阈值找话题切换点
# 语料取 data/星云智能产品手册.md 的两节正文拼起来：「版本与套餐」（讲价格）+「部署方式」（讲部署），
# 中间正好有一次真实的话题切换，用来检验断点检测是不是真能把话题边界找出来。
# 相似度的语义：embed() 返回的已是 L2 归一化向量，所以点积就是余弦；相似度骤降 = 话题切换 = 切点。
import re
from pathlib import Path
import numpy as np

def load_sentences(md_path, sections):
    """取指定小节的正文并切成句子；Markdown 表格行不是句子，先剔除（否则一整行表格会被当成一句）"""
    md = Path(md_path).read_text(encoding='utf-8')
    sents = []
    for name in sections:
        body = re.search(r'## %s\n(.*?)(?=\n## |\Z)' % name, md, re.S).group(1)
        body = '\n'.join(l for l in body.splitlines() if not l.strip().startswith('|'))
        sents += [s.strip().replace('\n', ' ') for s in re.split(r'(?<=[。！？])', body) if s.strip()]
    return sents

def semantic_breakpoints(sim_scores, threshold=0.6):
    """给定相邻句子的相似度序列，返回应切开的位置（相似度骤降处 = 话题切换点）"""
    return [i for i, s in enumerate(sim_scores) if s < threshold]

SRC, SECS = 'data/星云智能产品手册.md', ['版本与套餐', '部署方式']
SENTS = load_sentences(SRC, SECS)
BOUNDARY = len(load_sentences(SRC, [SECS[0]])) - 1     # 两小节之间的真实边界，落在第几条相邻句对上
print('语料：%s 的「%s」+「%s」，共 %d 句' % (SRC, SECS[0], SECS[1], len(SENTS)))
for i, s in enumerate(SENTS):
    print('  S%d: %s' % (i + 1, s[:38]))
print('真实小节边界：第 %d 条相邻句对（S%d 与 S%d 之间）' % (BOUNDARY + 1, BOUNDARY + 1, BOUNDARY + 2))

if _HAS_KEY:
    V = embed(SENTS)                     # 真调 text-embedding-v3；逐句向量化，成本随句数线性增长
    sims = [float(np.dot(V[i], V[i + 1])) for i in range(len(SENTS) - 1)]   # 归一化向量：点积即余弦
    cuts = semantic_breakpoints(sims, threshold=0.6)
    print('\n相邻句余弦相似度:', [round(s, 3) for s in sims])
    print('阈值 0.6 判定的话题切换点(相邻句对下标): %s' % cuts)
    print('→ 在 S%d|S%d 之间切开：S1-S2 同讲套餐（%.3f），S2→S3 骤降到 %.3f，'
          'S3-S5 又稳定在 %.3f 上下（同讲部署）。'
          % (cuts[0] + 1, cuts[0] + 2, sims[0], sims[cuts[0]], max(sims[2:])))
else:
    recorded("""相邻句余弦相似度: [0.617, 0.499, 0.773, 0.776, 0.703]
阈值 0.6 判定的话题切换点(相邻句对下标): [1]
→ 在 S2|S3 之间切开：S1-S2 同讲套餐（0.617），S2→S3 骤降到 0.499，S3-S5 又稳定在 0.776 上下（同讲部署）。""",
             '录制于 2026-09-12，模型 text-embedding-v3')
    sims = [0.617, 0.499, 0.773, 0.776, 0.703]           # 与上面录制内容一致的真实数值
    cuts = semantic_breakpoints(sims, threshold=0.6)

# 切点对不对，用真实的小节边界来验证（这是「标注」，不是模型给的）
print('\n校验：检出切点 %s vs 真实小节边界 %s → %s'
      % (cuts, [BOUNDARY], '完全吻合' if cuts == [BOUNDARY] else '不吻合，需要调阈值/换模型'))
# 绝对阈值 0.6 是「模型相关」的：text-embedding-v3 同话题相似度偏高，所以 0.6 落在同话题带下方；
# 换个模型/换批语料就得重标，用分布自适应的阈值可以免去手工调参——这里两者结果一致，可以互相印证。
_auto = float(np.mean(sims) - np.std(sims))
print('稳健性：换「均值-标准差」自适应阈值 %.3f，切点 %s —— 与固定 0.6 一致。'
      % (_auto, semantic_breakpoints(sims, _auto)))
print('\n→ 这就是 Semantic Chunking 的真实实现：逐句 embed → 相邻余弦 → 阈值切点，'
      '切出的块正好「一个话题一块」；07 课的长度切分只保证长度合适，这里保证语义内聚。')


In [ ]:
# 知识点·真调说明：Proposition Chunking —— 让 LLM 把一段长文拆成“一条只含一个断言”的最小事实陈述
# 这类 chunk 由 LLM 生成（而非按长度切），专为高精度、细粒度的问答服务。
_llm_live(
    prompt="""请把下面这段话拆成若干条“最小事实陈述”。要求：每条有且只有一个断言，不推理、不合并、不增删信息，输出为编号列表。
文本：缓存穿透指查询一个不存在的数据，由于缓存里也没有，请求会直接打到数据库。缓解手段有两种：一是布隆过滤器，能提前拦下大多数不存在的 key；二是缓存空值，把空结果也缓存一小段时间。注意布隆过滤器只能降低概率，不能完全杜绝穿透。""",
    system='你是文本切分专家，只输出编号列表，每条一句话，不要输出任何解释。',
    fallback="""1. 缓存穿透指查询一个不存在的数据，缓存中没有时请求会直接打到数据库。
2. 缓解缓存穿透的手段有两种。
3. 手段一是布隆过滤器，能提前拦下大多数不存在的 key。
4. 手段二是缓存空值，把空结果也缓存一小段时间。
5. 布隆过滤器只能降低概率，不能完全杜绝穿透。""",
    temperature=0.1,
)
print('→ LLM 拆出的每一条都是“独立可检索”的原子事实：入库时一条事实对应一个向量，问题命中哪条就精确召回哪条，而不必连带整段。')

In [ ]:
# 知识点·真调说明：Contextual Chunking —— 同一个小碎片，不加上下文 vs 加一句“它在讲什么”，模型回答天差地别
# 用 LLM 为被切下的碎片补一段上下文说明并随块入库，是 24 课 Contextual Retrieval 的思路。
print('① 只有碎片本身（缺上下文）—— 模型看不出这段在讲什么')
_llm_live(
    prompt="""请只依据下面给出的资料回答：这段文字在讲什么系统？
资料：「qps 从 8000 提到 20000」。""",
    system='你是问答助手，只能依据给定资料作答；资料信息不足时明确说“无法从资料判断”，不要自行脑补。',
    fallback="""无法从资料判断：只给出“qps 从 8000 提到 20000”一个数字，既看不出是什么系统，也看不出是哪个环节的指标。""",
    temperature=0.1,
)
print()
print('② 给碎片补上“这段属于哪份文档、在讲什么”的上下文说明再问 —— 能答了')
_llm_live(
    prompt="""请依据下面给出的资料回答：这段在讲什么系统、提升了什么？
资料（含上下文说明）：「本节选自《星云客服机器人 v3 技术方案》，主题是把消息队列从 Redis 换成自研存储并调优；其中写道：qps 从 8000 提到 20000」。""",
    system='你是问答助手，只能依据给定资料作答。',
    fallback="""在讲星云客服机器人 v3 的消息队列升级：把消息队列从 Redis 换成自研存储并调优后，qps 从 8000 提升到 20000。""",
    temperature=0.1,
)
print()
print('→ 碎片缺上下文时，即使被检索命中也无法正确作答；Contextual Chunking 用 LLM 给每个碎片补一句“它在讲什么”，让孤立数字也能被可靠回答。')

## 3. 结构类切分（按文档自身结构走）

| 方法 | 思路 | 适配文档 |
|------|------|---------|
| **Markdown Chunking** | 按 `# / ## / ###` 标题层级切，标题进 chunk | MD 文档 |
| **Code Chunking** | 按函数/类(甚至 AST)切，保持代码可执行语义 | 代码仓库（第 32 课） |
| **Table Chunking** | 表头+行成对保留，别把表拆碎 | Excel/表格 |
| **Hierarchical / Parent-Child** | 两级结构：大块(父)与小块(子)并存 | 长章节 |
| **Document Structure** | 结合第 5 课解析出的 标题/章节/页码 切 | PDF 手册 |

共同点：**利用文档自带的层级做切分锚点**，而不是按字数硬切——chunk 与原文结构对齐后，来源标注也更自然。

## 小结

- **语义类**切分：让一块讲一件事（Semantic/Proposition/Contextual…）；
- **结构类**切分：顺着标题/章节/表格/代码结构切；
- 两者可组合，且都要配合 **metadata**（下一课）才能真正好用。